In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import random
from collections import Counter
from nltk.translate.bleu_score import sentence_bleu
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
import math
import gc
gc.collect()

0

In [3]:

torch.cuda.empty_cache()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
# drive.mount('/content/drive')

Using device: cuda


In [24]:

PAD_TOKEN = 0
START_TOKEN = 1
END_TOKEN = 2
UNK_TOKEN = 3

HIDDEN_SIZE = 100
NUM_LAYERS = 3

LEARNING_RATE = 0.001
BATCH_SIZE = 32
EPOCHS = 10
EMBEDDING_DIM =200



HINDI_FASTTEXT ='/kaggle/input/datasets/kgcodex/embeddings/wiki.hi.vec'
ENGLISH_GLOVE = '/kaggle/input/datasets/kgcodex/embeddings/glove.6B.200d.txt' # Example: Hindi

GLOVE_DIM=200
FASTEXT_DIM=300

In [32]:
def train_bpe_tokenizer(sentences, vocab_size=25000):
   
    tokenizer = Tokenizer(BPE(unk_token="<unk>"))
    tokenizer.pre_tokenizer = Whitespace()

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["<pad>", "<start>", "<end>", "<unk>"]
    )
    

    tokenizer.train_from_iterator(sentences, trainer)
    return tokenizer

In [33]:

train_df = pd.read_csv('/kaggle/input/datasets/kgcodex/machine-translation-task/team10_hi_train.csv').dropna()
valid_df = pd.read_csv('/kaggle/input/datasets/kgcodex/machine-translation-task/team10_hi_valid.csv').dropna()
test_df = pd.read_csv('/kaggle/input/datasets/kgcodex/machine-translation-task/team10_hi_test.csv').dropna()


eng_tokenizer = train_bpe_tokenizer(train_df['source'].astype(str).tolist())
ind_tokenizer = train_bpe_tokenizer(train_df['target'].astype(str).tolist())


SRC_VOCAB_SIZE = eng_tokenizer.get_vocab_size()
TGT_VOCAB_SIZE = ind_tokenizer.get_vocab_size()

print(f"English Vocab Size: {SRC_VOCAB_SIZE} | Indian Vocab Size: {TGT_VOCAB_SIZE}")







English Vocab Size: 25000 | Indian Vocab Size: 25000


In [34]:
class TranslationDataset(Dataset):
    def __init__(self, df, eng_tokenizer, ind_tokenizer):
        self.df = df
        self.eng_tokenizer = eng_tokenizer
        self.ind_tokenizer = ind_tokenizer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
       
        src_sent = str(self.df.iloc[idx]['source'])
        trg_sent = str(self.df.iloc[idx]['target'])

      
   
        en_encoding = self.eng_tokenizer.encode(src_sent).ids
        in_encoding = self.ind_tokenizer.encode(trg_sent).ids

       
        in_indices = [1] + in_encoding + [2]

        return torch.tensor(en_encoding), torch.tensor(in_indices)

def padding_fn(batch):
    en_list, in_list = [], []

    for en_item, in_item in batch:
        en_list.append(en_item)
        in_list.append(in_item)


    en_padded = pad_sequence(en_list, padding_value=0, batch_first=False)
    in_padded = pad_sequence(in_list, padding_value=0, batch_first=False)

    return en_padded, in_padded

In [98]:

train_dataset = TranslationDataset(train_df, eng_tokenizer, ind_tokenizer)
valid_dataset = TranslationDataset(valid_df, eng_tokenizer, ind_tokenizer)
test_dataset = TranslationDataset(test_df, eng_tokenizer, ind_tokenizer)


train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=padding_fn)
valid_loader = DataLoader(valid_dataset, batch_size=16, shuffle=False, collate_fn=padding_fn)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, collate_fn=padding_fn)


print(f"BPE Train Batches: {len(train_loader)}")

BPE Train Batches: 4375


In [65]:


def get_emb_matrix(file_path, tokenizer, dim, is_fasttext=True):

    vocab_dict = tokenizer.get_vocab()
    vocab_size = len(vocab_dict)

    matrix = np.random.normal(scale=0.1, size=(vocab_size, dim))
 
    matrix[0] = np.zeros(dim)

    print(f"Loading embeddings from {file_path}...")
    found_count = 0
    
    with open(file_path, 'r', encoding='utf-8') as f:
      
        if is_fasttext:
            f.readline()

        for line in f:
            parts = line.split()
            if len(parts) <= dim:
                continue

            word = parts[0]
           
            if word in vocab_dict:
                try:
                    idx = vocab_dict[word]
                    vector = np.array(parts[1:dim+1], dtype=np.float32)
                    matrix[idx] = vector
                    found_count += 1
                except ValueError:
                    continue

    print(f"Matched {found_count} out of {vocab_size} BPE tokens.")
    return torch.tensor(matrix).float()

In [66]:
# 1. Load English GloVe matrix using the BPE tokenizer

en_matrix = get_emb_matrix(
    ENGLISH_GLOVE, 
    eng_tokenizer, 
    GLOVE_DIM, 
    is_fasttext=False
)

# 2. Load Hindi FastText matrix using the BPE tokenizer

ind_matrix = get_emb_matrix(
    HINDI_FASTTEXT, 
    ind_tokenizer, 
    FASTEXT_DIM, 
    is_fasttext=True
)


print(f"English Matrix Shape: {en_matrix.shape}") # Should be (25000, 200)
print(f"Indian Matrix Shape: {ind_matrix.shape}")   # Should be (25000, 300)

Loading embeddings from /kaggle/input/datasets/kgcodex/embeddings/glove.6B.200d.txt...
Matched 13563 out of 25000 BPE tokens.
Loading embeddings from /kaggle/input/datasets/kgcodex/embeddings/wiki.hi.vec...
Matched 17791 out of 25000 BPE tokens.
English Matrix Shape: torch.Size([25000, 200])
Indian Matrix Shape: torch.Size([25000, 300])


In [69]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        
     
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
      
        pe = pe.unsqueeze(0).transpose(0, 1)  
        self.register_buffer('pe', pe)

    def forward(self, x):
      
        x = x + self.pe[:x.size(0), :, :]
        return self.dropout(x)


class Transformer_Translator(nn.Module):
    def __init__(self, src_vocab_size, trg_vocab_size, src_pad_idx, embed_dim=512, nhead=8, num_layers=3):
        super().__init__()
        self.src_pad_idx = src_pad_idx
        
      
        self.src_embedding = nn.Sequential(
            nn.Embedding.from_pretrained(en_matrix, freeze=False),
            nn.Linear(en_matrix.shape[1], embed_dim)   
        )
        
     
        self.trg_embedding = nn.Sequential(
            nn.Embedding.from_pretrained(ind_matrix, freeze=False),
            nn.Linear(ind_matrix.shape[1], embed_dim)  
        )
        
        self.positional_encoding = PositionalEncoding(embed_dim)
 
        self.transformer = nn.Transformer(
            d_model=embed_dim,
            nhead=nhead,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=2048,
            dropout=0.1
        )
        
      
        self.fc_out = nn.Linear(embed_dim, trg_vocab_size)

    def make_src_mask(self, src):
      
        return (src == self.src_pad_idx).transpose(0, 1) 

    def forward(self, src, trg):
  
        src_seq_len, batch_size = src.shape
        trg_seq_len, _ = trg.shape
        
       
        trg_mask = self.transformer.generate_square_subsequent_mask(trg_seq_len).to(src.device)
        src_padding_mask = self.make_src_mask(src)
        
      
        src_emb = self.positional_encoding(self.src_embedding(src))
        trg_emb = self.positional_encoding(self.trg_embedding(trg))
        
   
        output = self.transformer(
            src_emb,
            trg_emb,
            src_key_padding_mask=src_padding_mask,
            tgt_mask=trg_mask
        )
        
        return self.fc_out(output)

In [82]:
# Use the trained BPE tokenizer lengths instead of the old vocab dictionaries
model = Transformer_Translator(
    src_vocab_size=eng_tokenizer.get_vocab_size(), 
    trg_vocab_size=ind_tokenizer.get_vocab_size(), 
    src_pad_idx=0,           
    embed_dim=512,          
    nhead=8,               
    num_layers=3             
).to(device)


optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)


criterion = nn.CrossEntropyLoss(ignore_index=0)

In [74]:


def train_model(model, train_loader, valid_loader, optimizer, criterion, epochs, device):
    
    scaler = torch.amp.GradScaler('cuda') 
    
   

    print(f"Starting Training for {epochs} epochs...")

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0
        
        for batch_idx, (src, trg) in enumerate(train_loader):
            # Move data to GPU
            src, trg = src.to(device), trg.to(device)
            optimizer.zero_grad()

           
            with torch.amp.autocast('cuda'):
                
                output = model(src, trg[:-1])
                
            
                output_reshaped = output.view(-1, output.shape[-1])
                target_reshaped = trg[1:].view(-1)
                
                loss = criterion(output_reshaped, target_reshaped)

     
            scaler.scale(loss).backward()
            
           
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
          
            scaler.step(optimizer)
            scaler.update()

            total_train_loss += loss.item()

            if batch_idx % 500 == 0:
                print(f"Epoch [{epoch+1}/{epochs}] | Batch {batch_idx} | Loss: {loss.item():.4f}")

           
            del src, trg, output, loss, output_reshaped, target_reshaped

       
        torch.cuda.empty_cache()
        gc.collect()

        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for src, trg in valid_loader:
                src, trg = src.to(device), trg.to(device)
                
                with torch.amp.autocast('cuda'):
                    output = model(src, trg[:-1])
                    val_loss = criterion(output.view(-1, output.shape[-1]), trg[1:].view(-1))
                
                total_val_loss += val_loss.item()
                del src, trg, output, val_loss

        avg_train_loss = total_train_loss / len(train_loader)
        avg_val_loss = total_val_loss / len(valid_loader)
        
        print(f"\n==> Epoch [{epoch+1}/{epochs}] Summary:")
        print(f"    Avg Train Loss: {avg_train_loss:.4f} | Avg Val Loss: {avg_val_loss:.4f}")
        print("-" * 35)

    print("Training Complete!")

In [83]:

torch.cuda.empty_cache()
gc.collect()
import os

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
train_model(
    model=model, 
    train_loader=train_loader, 
    valid_loader=valid_loader,
    optimizer=optimizer, 
    criterion=criterion, 
    epochs=20,
    device=device
)

Starting Training for 20 epochs...
Epoch [1/20] | Batch 0 | Loss: 10.3709
Epoch [1/20] | Batch 500 | Loss: 6.4510
Epoch [1/20] | Batch 1000 | Loss: 6.5500
Epoch [1/20] | Batch 1500 | Loss: 6.8414
Epoch [1/20] | Batch 2000 | Loss: 5.9942
Epoch [1/20] | Batch 2500 | Loss: 5.9540
Epoch [1/20] | Batch 3000 | Loss: 5.4665
Epoch [1/20] | Batch 3500 | Loss: 5.4100
Epoch [1/20] | Batch 4000 | Loss: 5.8125

==> Epoch [1/20] Summary:
    Avg Train Loss: 6.1972 | Avg Val Loss: 5.6233
-----------------------------------
Epoch [2/20] | Batch 0 | Loss: 5.6255
Epoch [2/20] | Batch 500 | Loss: 5.9399
Epoch [2/20] | Batch 1000 | Loss: 5.3210
Epoch [2/20] | Batch 1500 | Loss: 5.4541
Epoch [2/20] | Batch 2000 | Loss: 5.4155
Epoch [2/20] | Batch 2500 | Loss: 5.1630
Epoch [2/20] | Batch 3000 | Loss: 5.2617
Epoch [2/20] | Batch 3500 | Loss: 5.1172
Epoch [2/20] | Batch 4000 | Loss: 5.4857

==> Epoch [2/20] Summary:
    Avg Train Loss: 5.4658 | Avg Val Loss: 5.2508
-----------------------------------
Epoch [3

In [102]:
def translate_batch_beam(model, src_batch, ind_tokenizer, beam_size=3, max_len=100):
    model.eval()
    batch_size = src_batch.shape[1]
    device = src_batch.device
    
    with torch.no_grad():
       
        src_padding_mask = model.make_src_mask(src_batch)
        src_emb = model.positional_encoding(model.src_embedding(src_batch))
        memory = model.transformer.encoder(src_emb, src_key_padding_mask=src_padding_mask)

       
        all_final_ids = []

        for b in range(batch_size):
           
            curr_memory = memory[:, b:b+1, :]
            curr_mask = src_padding_mask[b:b+1, :]
            
          
            beams = [([1], 0.0)] 
            
            for _ in range(max_len):
                candidates = []
                for seq, score in beams:
                    if seq[-1] == 2: 
                        candidates.append((seq, score))
                        continue
                    
                    trg_tensor = torch.LongTensor(seq).unsqueeze(1).to(device)
                    trg_mask = model.transformer.generate_square_subsequent_mask(len(seq)).to(device)
                    trg_emb = model.positional_encoding(model.trg_embedding(trg_tensor))
                    
                    output = model.transformer.decoder(trg_emb, curr_memory, tgt_mask=trg_mask, memory_key_padding_mask=curr_mask)
                    logits = torch.log_softmax(model.fc_out(output[-1, :, :]), dim=-1)
                    
              
                    top_probs, top_indices = logits.topk(beam_size)
                    for i in range(beam_size):
                        candidates.append((seq + [top_indices[0, i].item()], score + top_probs[0, i].item()))
                
         
                beams = sorted(candidates, key=lambda x: x[1], reverse=True)[:beam_size]
                
                if all(s[-1] == 2 for s, _ in beams):
                    break
            
   
            best_seq = beams[0][0]
            all_final_ids.append([idx for idx in best_seq if idx not in [0, 1, 2]])

    return [ind_tokenizer.decode(ids) for ids in all_final_ids]

In [105]:
def calculate_bleu_batch(data_loader, model, ind_tokenizer):
    targets = []
    predictions = []

    model.eval()

    print(f"Total Batches: {len(data_loader)}")

    with torch.no_grad():
        for cnt, (src, trg) in enumerate(data_loader):
            src = src.to(device)
            
          
            batch_preds = translate_batch_beam(model, src, ind_tokenizer, beam_size=3, max_len=50)
            
          
            for pred in batch_preds:
                predictions.append(pred.split())

           
            trg_batch = trg.cpu().numpy().T 
            for sentence_indices in trg_batch:
                clean_indices = [idx for idx in sentence_indices if idx not in [0, 1, 2]]
                ref_sent = ind_tokenizer.decode(clean_indices)
                targets.append([ref_sent.split()])
            
            if cnt % 5 == 0:
                print(f"Batch: {cnt}/{len(data_loader)} processed...")


    b1 = corpus_bleu(targets, predictions, weights=(1, 0, 0, 0))
    b2 = corpus_bleu(targets, predictions, weights=(0.5, 0.5, 0, 0))
    b3 = corpus_bleu(targets, predictions, weights=(0.33, 0.33, 0.33, 0))
    b4 = corpus_bleu(targets, predictions, weights=(0.25, 0.25, 0.25, 0.25))

    print(f"\nFinal BLEU Scores:")
    print(f"BLEU-1: {b1*100:.2f} | BLEU-2: {b2*100:.2f} | BLEU-3: {b3*100:.2f} | BLEU-4: {b4*100:.2f}")
    
    return b1, b2, b3, b4

In [106]:

calculate_bleu_batch(test_loader, model, ind_tokenizer)

Total Batches: 79
Batch: 0/79 processed...
Batch: 5/79 processed...
Batch: 10/79 processed...
Batch: 15/79 processed...
Batch: 20/79 processed...
Batch: 25/79 processed...
Batch: 30/79 processed...
Batch: 35/79 processed...
Batch: 40/79 processed...
Batch: 45/79 processed...
Batch: 50/79 processed...
Batch: 55/79 processed...
Batch: 60/79 processed...
Batch: 65/79 processed...
Batch: 70/79 processed...
Batch: 75/79 processed...

Final BLEU Scores:
BLEU-1: 30.17 | BLEU-2: 19.14 | BLEU-3: 12.84 | BLEU-4: 8.48


(0.30165820131494686,
 0.19138526046839704,
 0.12843509606796938,
 0.08478512762184101)

In [111]:
import random

def generate_text_report(model, dataset, eng_tokenizer, ind_tokenizer, device, num_examples=5, beam_size=3):
    """
    Randomly samples examples from the dataset and prints Source, Ground Truth, and Model Output.
    """
    model.eval()
    
   
    indices = random.sample(range(len(dataset)), num_examples)
    
    print(f"\n{'='*25} TRANSLATION REPORT {'='*25}")
    
    for i, idx in enumerate(indices):
   
        src_tensor, trg_tensor = dataset[idx]
        
        src_sentence = eng_tokenizer.decode(src_tensor.tolist())
        
        clean_trg_ids = [tid.item() for tid in trg_tensor if tid.item() not in [0, 1, 2]]
        gt_sentence = ind_tokenizer.decode(clean_trg_ids)
        
        src_batch = src_tensor.unsqueeze(1).to(device)
        
      
        prediction = translate_batch_beam(model, src_batch, ind_tokenizer, beam_size=beam_size, max_len=50)[0] 
        
        print(f"\n--- Example {i+1} (Index: {idx}) ---")
        print(f"Source (EN)      : {src_sentence}")
        print(f"Ground Truth (HI): {gt_sentence}")
        print(f"Model Prediction : {prediction}")
    
    print(f"\n{'='*64}")

print("Generating Report for Training Data...")
generate_text_report(model, train_dataset, eng_tokenizer, ind_tokenizer, device, num_examples=5)

print("\nGenerating Report for Test Data...")
generate_text_report(model, test_dataset, eng_tokenizer, ind_tokenizer, device, num_examples=5)

Generating Report for Training Data...

========================= TRANSLATION REPORT =========================

--- Example 1 (Index: 38818) ---
Source (EN)      : " India shall forever remain the home for persecut ed Hindus , irrespective of your opposition Sir ."""
Ground Truth (HI): भारत सदैव सताए हुए हिंदुओं का घर रहेगा , आपके विरोध के बावजूद सर ।
Model Prediction : भारत हमेशा हिंदू राष्ट्र के लिए घर नहीं रहेगा , आपके विरोध के बावजूद भारत हमेशा रहेंगे ।

--- Example 2 (Index: 36081) ---
Source (EN)      : Sup porters of Congress MLA from Bh or , San gram Th op te , allegedly went on a ramp age at the party headquarters in Pune on Tuesday .
Ground Truth (HI): भो र के कांग्रेस के विधायक संग्राम थो प्ट े के समर्थकों ने उन्हें महाराष्ट्र मंत्रिमंडल में शामिल नहीं किये जाने को लेकर मंगलवार को पुणे में पार्टी कार्यालय पर हमला किया ।
Model Prediction : पुणे के कांग्रेस विधायक और कांग्रेस के कार्यकर्ताओं ने मंगलवार को पुणे स्थित पार्टी मुख्यालय में पार्टी मुख्यालय पर एक व्यक्ति की उम्र में